# Text Summarization using a Transformer Model (T5)

**Aim:** To summarise news articles using a pre-trained Transformer (T5-small) that is fine-tuned on a small part
of the CNN/DailyMail dataset, to compare it with a simple extractive method, and to evaluate the summaries with
ROUGE.

**Steps:** load data → tokenize → fine-tune T5-small → generate abstractive summaries → extractive summary →
**test** (ROUGE comparison of four methods, per-article scores, best and worst examples, custom article).

*Run on a GPU:* Runtime → Change runtime type → T4 GPU.

In [ ]:
!pip install -q -U transformers datasets evaluate sentencepiece rouge_score

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import re
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import Counter
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
import evaluate

device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch version:", torch.__version__)
print("Using device   :", device)
if torch.cuda.is_available():
    print("GPU            :", torch.cuda.get_device_name(0))

## 1. Load and inspect the dataset

In [ ]:
# [LOAD]
dataset = load_dataset("abisee/cnn_dailymail", "3.0.0")
print(dataset)

print("\nARTICLE:")
print(dataset["train"][0]["article"][:700], "...")
print("\n" + "=" * 80)
print("REFERENCE SUMMARY:")
print(dataset["train"][0]["highlights"])

In [ ]:
# A small part of the data keeps the run short on Colab
train_size, validation_size, test_size = 2000, 200, 100

train_dataset = dataset["train"].select(range(train_size))
validation_dataset = dataset["validation"].select(range(validation_size))
test_dataset = dataset["test"].select(range(test_size))

print("Training samples  :", len(train_dataset))
print("Validation samples:", len(validation_dataset))
print("Test samples      :", len(test_dataset))

# length statistics
words_article = [len(a.split()) for a in train_dataset["article"]]
words_summary = [len(h.split()) for h in train_dataset["highlights"]]
print(f"\nAverage article length: {np.mean(words_article):.0f} words")
print(f"Average summary length: {np.mean(words_summary):.0f} words")
print(f"Typical compression   : the summary is about {np.mean(words_summary) / np.mean(words_article) * 100:.0f}% of the article")

## 2. Load T5-small and tokenize the data

In [ ]:
model_name = "t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model.to(device)
print("T5-small loaded successfully on", device)

prefix = "summarize: "
max_input_length = 512
max_target_length = 128

In [ ]:
def preprocess_function(examples):
    inputs = [prefix + article for article in examples["article"]]
    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True)

    labels = tokenizer(text_target=examples["highlights"], max_length=max_target_length, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_dataset.map(preprocess_function, batched=True,
                                    remove_columns=train_dataset.column_names)
tokenized_validation = validation_dataset.map(preprocess_function, batched=True,
                                              remove_columns=validation_dataset.column_names)
print(tokenized_train)

print("\nInput IDs:", tokenized_train[0]["input_ids"][:20])
print("Labels   :", tokenized_train[0]["labels"][:20])

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

## 3. ROUGE metric and fine-tuning

In [ ]:
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    if isinstance(predictions, tuple):
        predictions = predictions[0]
    predictions = np.asarray(predictions)
    if predictions.ndim == 3:                       # logits -> token ids
        predictions = np.argmax(predictions, axis=-1)
    predictions = predictions.astype(np.int64)

    labels = np.where(np.asarray(labels) == -100, tokenizer.pad_token_id, labels).astype(np.int64)

    decoded_predictions = tokenizer.batch_decode(predictions, skip_special_tokens=True,
                                                 clean_up_tokenization_spaces=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True,
                                            clean_up_tokenization_spaces=True)

    result = rouge.compute(predictions=decoded_predictions, references=decoded_labels)
    return {key: round(value, 4) for key, value in result.items()}

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./t5_summarization",
    eval_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=2,
    save_strategy="epoch",
    predict_with_generate=True,          # generate summaries during evaluation
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Starting T5 fine-tuning...")
trainer.train()
print("\nTraining completed successfully.")

In [ ]:
evaluation_results = trainer.evaluate()
print("Validation results:")
for key, value in evaluation_results.items():
    print(f"  {key}: {value}")

# Save and reload the fine-tuned model
save_path = "./t5_summarization_model"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print("\nModel saved at:", save_path)

summarizer_tokenizer = AutoTokenizer.from_pretrained(save_path)
summarizer_model = AutoModelForSeq2SeqLM.from_pretrained(save_path)
summarizer_model.to(device)
summarizer_model.eval()
print("Fine-tuned model loaded successfully.")

## 4. Abstractive summarization (generate new sentences)

In [ ]:
def generate_summary(article):
    input_text = "summarize: " + article
    inputs = summarizer_tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True)
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        summary_ids = summarizer_model.generate(
            **inputs,
            max_length=128,
            min_length=30,
            num_beams=4,
            length_penalty=2.0,
            early_stopping=True
        )
    return summarizer_tokenizer.decode(summary_ids[0], skip_special_tokens=True)

article = test_dataset[0]["article"]
reference_summary = test_dataset[0]["highlights"]
generated_summary = generate_summary(article)

print("=" * 80)
print("ORIGINAL ARTICLE (first 800 characters)")
print("=" * 80)
print(article[:800], "...")
print("\n" + "=" * 80)
print("GENERATED ABSTRACTIVE SUMMARY")
print("=" * 80)
print(generated_summary)
print("\n" + "=" * 80)
print("REFERENCE SUMMARY")
print("=" * 80)
print(reference_summary)

## 5. Extractive summarization (pick the best existing sentences)

In [ ]:
def split_sentences(text):
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in sentences if s.strip()]

def extractive_summary(text, num_sentences=3):
    sentences = split_sentences(text)
    if len(sentences) <= num_sentences:
        return " ".join(sentences)

    # word frequencies (ignoring very short words)
    words = re.findall(r'\b[a-zA-Z]{4,}\b', text.lower())
    frequency = Counter(words)

    # score a sentence by the average frequency of its words
    scores = []
    for i, sentence in enumerate(sentences):
        sentence_words = re.findall(r'\b[a-zA-Z]{4,}\b', sentence.lower())
        score = sum(frequency[w] for w in sentence_words) / (len(sentence_words) + 1)
        scores.append((score, i))

    # keep the best sentences, but print them in their original order
    best = sorted(sorted(scores, reverse=True)[:num_sentences], key=lambda x: x[1])
    return " ".join(sentences[i] for _, i in best)

def lead_3(text):
    # simple but strong baseline for news: the first three sentences
    return " ".join(split_sentences(text)[:3])

print("EXTRACTIVE SUMMARY (3 best sentences):")
print(extractive_summary(article))
print("\nLEAD-3 BASELINE (first 3 sentences):")
print(lead_3(article))

---
# Testing the summarizers

Four methods are compared on the **100 test articles** (never used for training):

| Method | What it does |
|---|---|
| **Lead-3** | takes the first three sentences |
| **Extractive** | picks the three sentences with the most frequent words |
| **T5-small (not fine-tuned)** | the original pre-trained model |
| **T5-small (fine-tuned)** | the model trained above |

*ROUGE-1 / ROUGE-2* count matching words / word pairs with the reference summary, *ROUGE-L* the longest common
sequence. Higher is better.

In [ ]:
def generate_summaries(model_, tokenizer_, articles, batch_size=8):
    """Summarise many articles at once (much faster than one by one)."""
    model_.eval()
    outputs = []
    for i in range(0, len(articles), batch_size):
        batch = [prefix + a for a in articles[i:i + batch_size]]
        enc = tokenizer_(batch, return_tensors="pt", max_length=512, truncation=True, padding=True).to(device)
        with torch.no_grad():
            ids = model_.generate(**enc, max_length=128, min_length=30, num_beams=4,
                                  length_penalty=2.0, early_stopping=True)
        outputs.extend(tokenizer_.batch_decode(ids, skip_special_tokens=True))
    return outputs

test_articles = list(test_dataset["article"])
test_references = list(test_dataset["highlights"])

print("Generating summaries for", len(test_articles), "test articles ...")
predictions = {
    "Lead-3": [lead_3(a) for a in test_articles],
    "Extractive (word frequency)": [extractive_summary(a) for a in test_articles],
}

base_model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
predictions["T5-small (not fine-tuned)"] = generate_summaries(base_model, tokenizer, test_articles)
predictions["T5-small (fine-tuned)"] = generate_summaries(summarizer_model, summarizer_tokenizer, test_articles)
print("Done.")

## Test 1: ROUGE comparison of the four methods

In [ ]:
rows = []
for name, preds in predictions.items():
    r = rouge.compute(predictions=preds, references=test_references, use_stemmer=True)
    rows.append({
        "Method": name,
        "ROUGE-1": round(r["rouge1"], 4),
        "ROUGE-2": round(r["rouge2"], 4),
        "ROUGE-L": round(r["rougeL"], 4),
        "Avg words": round(np.mean([len(p.split()) for p in preds]), 1),
    })

rouge_df = pd.DataFrame(rows).set_index("Method")
rouge_df.loc["Reference summaries", "Avg words"] = round(np.mean([len(r.split()) for r in test_references]), 1)
display(rouge_df)

print("Best method by ROUGE-L:", rouge_df["ROUGE-L"].dropna().idxmax())

rouge_df.dropna(subset=["ROUGE-1"])[["ROUGE-1", "ROUGE-2", "ROUGE-L"]].plot(kind="bar", figsize=(10, 5))
plt.title("ROUGE scores on 100 unseen test articles")
plt.ylabel("Score")
plt.xticks(rotation=15)
plt.show()

print("Note: the ROUGE printed during training is lower because the Trainer generates short summaries")
print("by default. Here every model uses the same generation settings (up to 128 tokens, 4 beams).")

## Test 2: per-article scores of the fine-tuned model

In [ ]:
best_name = "T5-small (fine-tuned)"
per_article = rouge.compute(predictions=predictions[best_name], references=test_references,
                            use_stemmer=True, use_aggregator=False)
scores_l = np.array(per_article["rougeL"])

print(f"ROUGE-L over {len(scores_l)} articles:  mean {scores_l.mean():.3f}   median {np.median(scores_l):.3f}"
      f"   min {scores_l.min():.3f}   max {scores_l.max():.3f}")

plt.figure(figsize=(8, 4))
plt.hist(scores_l, bins=20)
plt.axvline(scores_l.mean(), color="red", linestyle="--", label=f"Mean {scores_l.mean():.3f}")
plt.xlabel("ROUGE-L of one article")
plt.ylabel("Number of articles")
plt.title("Distribution of per-article ROUGE-L (fine-tuned T5-small)")
plt.legend()
plt.show()

# length and compression of the generated summaries
lengths = pd.DataFrame({
    "Article words": [len(a.split()) for a in test_articles],
    "Generated words": [len(p.split()) for p in predictions[best_name]],
    "Reference words": [len(r.split()) for r in test_references],
})
lengths["Compression"] = (lengths["Generated words"] / lengths["Article words"] * 100).round(1)
display(lengths.describe().loc[["mean", "min", "max"]].round(1))

In [ ]:
order = np.argsort(scores_l)
def show_case(i, title):
    print("=" * 80)
    print(f"{title}   (ROUGE-L = {scores_l[i]:.3f})")
    print("=" * 80)
    print("ARTICLE  :", test_articles[i][:300].replace("\n", " "), "...")
    print("GENERATED:", predictions[best_name][i])
    print("REFERENCE:", test_references[i].replace("\n", " "))
    print()

show_case(order[-1], "BEST SUMMARY")
show_case(order[0], "WORST SUMMARY")

## Test 3: your own article

In [ ]:
custom_article = (
    "The city council approved a new plan on Tuesday to expand the metro network, adding three lines and twenty "
    "stations over the next eight years. Officials said the project will cost about two billion dollars and is "
    "expected to reduce traffic congestion by nearly fifteen percent. Residents in the eastern suburbs, who "
    "currently rely on crowded buses, welcomed the decision, although some groups raised concerns about noise "
    "and construction delays. The mayor said that work on the first line will begin early next year and that "
    "regular updates will be shared with the public. Construction is to be funded jointly by the city and the "
    "national government, with part of the money coming from a new transport levy."
)

print("ARTICLE (", len(custom_article.split()), "words )")
print(custom_article)
print("\n--- Fine-tuned T5-small ---")
fine_tuned_summary = generate_summaries(summarizer_model, summarizer_tokenizer, [custom_article])[0]
print(fine_tuned_summary)
print("\n--- Extractive ---")
print(extractive_summary(custom_article))

print(f"\nCompression: {len(fine_tuned_summary.split())} words out of {len(custom_article.split())}"
      f" ({len(fine_tuned_summary.split()) / len(custom_article.split()) * 100:.0f}%)")

## Conclusion

T5-small was fine-tuned on 2,000 CNN/DailyMail articles to produce abstractive summaries and compared with an
extractive method and the Lead-3 baseline. The testing section evaluated all methods on 100 unseen articles with
ROUGE-1, ROUGE-2 and ROUGE-L, studied the spread of per-article scores, showed the best and worst summaries and
tried a hand-written article. Lead-3 is a very strong baseline for news; a model trained on only a small part of
the data may not beat it, and more data and epochs would be the next step.